## Spectral cleaning of principal minor corruptions

We generate a GOE matrix $X$, corrupt it with an $\varepsilon$-principal
minor perturbation (supported on $|S| = \lfloor \varepsilon n \rfloor$
rows/columns), and run Algorithm 3.7 (spectral cleaning) to recover a
matrix $\hat{Y}$ with $\|\hat{Y}\|_{\mathrm{op}} \leq K = 5\,\mathbb{E}\|X\|_{\mathrm{op}}$.

In [ ]:
import matplotlib
matplotlib.use("Agg")

import grmdil.amp  # noqa: F401 — silence JAX GPU probe log before jax import

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

jax.config.update("jax_enable_x64", True)

from grmdil.amp.principal_minor import (
    op_norm_symmetric,
    spectral_cleaning,
    spectral_threshold_k,
)
from grmdil.amp.synthetic import goe_matrix, principal_minor_corruption

### Operator norm before and after cleaning across corruption strengths

In [ ]:
n = 400
eps = 0.05
strengths = [0, 2, 5, 10, 15, 20, 30]
k = spectral_threshold_k(5.0)

norms_before, norms_after, n_removed = [], [], []

for strength in strengths:
    x = goe_matrix(n, jax.random.PRNGKey(42))
    if strength == 0:
        y = x
    else:
        y, _ = principal_minor_corruption(x, jax.random.PRNGKey(43), eps, float(strength))
    y_hat, removed = spectral_cleaning(y, k, jax.random.PRNGKey(44), max_removals=n)
    norms_before.append(float(op_norm_symmetric(y)))
    norms_after.append(float(op_norm_symmetric(y_hat)))
    n_removed.append(removed.shape[0])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].bar(range(len(strengths)), norms_before, alpha=0.6, label=r"$\|Y\|_{\mathrm{op}}$ (corrupted)")
axes[0].bar(range(len(strengths)), norms_after, alpha=0.6, label=r"$\|\hat{Y}\|_{\mathrm{op}}$ (cleaned)")
axes[0].axhline(k, color="k", ls="--", label=f"$K = {k:.0f}$")
axes[0].set_xticks(range(len(strengths)))
axes[0].set_xticklabels([str(s) for s in strengths])
axes[0].set_xlabel("Corruption strength")
axes[0].set_ylabel("Operator norm")
axes[0].set_title(f"Spectral cleaning ($n={n}$, $\\varepsilon={eps}$)")
axes[0].legend(fontsize=8)

axes[1].bar(range(len(strengths)), n_removed, color="tab:orange")
axes[1].axhline(eps * n, color="k", ls="--", label=f"$\\varepsilon n = {eps * n:.0f}$")
axes[1].set_xticks(range(len(strengths)))
axes[1].set_xticklabels([str(s) for s in strengths])
axes[1].set_xlabel("Corruption strength")
axes[1].set_ylabel("Rows/columns removed")
axes[1].set_title("Number of removals")
axes[1].legend()

plt.tight_layout()
plt.savefig("../report/figures/02_spectral_cleaning.pdf", bbox_inches="tight")
plt.show()

### Eigenvalue spectrum before and after cleaning

In [ ]:
strength = 20
x = goe_matrix(n, jax.random.PRNGKey(50))
y, _ = principal_minor_corruption(x, jax.random.PRNGKey(51), eps, strength)
y_hat, removed = spectral_cleaning(y, k, jax.random.PRNGKey(52), max_removals=n)

evals_clean = jnp.linalg.eigh(x)[0]
evals_corrupt = jnp.linalg.eigh(y)[0]
evals_cleaned = jnp.linalg.eigh(y_hat)[0]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].hist(evals_clean, bins=80, density=True, alpha=0.4, label="Clean $X$")
axes[0].hist(evals_corrupt, bins=80, density=True, alpha=0.4, label=f"Corrupted $Y$ (strength={strength})")
axes[0].hist(evals_cleaned, bins=80, density=True, alpha=0.4, label=r"Cleaned $\hat{Y}$")
axes[0].axvline(k, color="k", ls="--", alpha=0.5, label=f"$K={k:.0f}$")
axes[0].axvline(-k, color="k", ls="--", alpha=0.5)
axes[0].set_xlabel("Eigenvalue")
axes[0].set_ylabel("Density")
axes[0].set_title("Full range: corruption outliers visible")
axes[0].legend(fontsize=7)

zoom = 5.0
mask_clean = jnp.abs(evals_clean) <= zoom
mask_corrupt = jnp.abs(evals_corrupt) <= zoom
mask_cleaned = jnp.abs(evals_cleaned) <= zoom
axes[1].hist(evals_clean[mask_clean], bins=60, density=True, alpha=0.4, label="Clean $X$")
axes[1].hist(evals_corrupt[mask_corrupt], bins=60, density=True, alpha=0.4, label=f"Corrupted $Y$")
axes[1].hist(evals_cleaned[mask_cleaned], bins=60, density=True, alpha=0.4, label=r"Cleaned $\hat{Y}$")
axes[1].axvline(k, color="k", ls="--", alpha=0.5)
axes[1].axvline(-k, color="k", ls="--", alpha=0.5)
xs = jnp.linspace(-2, 2, 200) # we overlay the Wigner semicircle for reference
axes[1].plot(xs, jnp.sqrt(jnp.maximum(4 - xs**2, 0)) / (2 * jnp.pi),
             "k-", lw=1.5, alpha=0.6, label="Semicircle law")
axes[1].set_xlim(-zoom, zoom)
axes[1].set_xlabel("Eigenvalue")
axes[1].set_ylabel("Density")
axes[1].set_title("Zoomed $[-5, 5]$: semicircle recovery")
axes[1].legend(fontsize=7)

fig.suptitle(f"Eigenvalue spectra ($n={n}$, $\\varepsilon={eps}$, strength$={strength}$)",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig("../report/figures/02_eigenvalue_spectra.pdf", bbox_inches="tight")

# Zoom-only figure for slides (omit full-range panel with extreme outliers).
fig_z, ax_z = plt.subplots(1, 1, figsize=(7.2, 4.2))
ax_z.hist(evals_clean[mask_clean], bins=60, density=True, alpha=0.4, label="Clean $X$")
ax_z.hist(evals_corrupt[mask_corrupt], bins=60, density=True, alpha=0.4, label=f"Corrupted $Y$")
ax_z.hist(evals_cleaned[mask_cleaned], bins=60, density=True, alpha=0.4, label=r"Cleaned $\hat{Y}$")
ax_z.axvline(k, color="k", ls="--", alpha=0.5)
ax_z.axvline(-k, color="k", ls="--", alpha=0.5)
ax_z.plot(xs, jnp.sqrt(jnp.maximum(4 - xs**2, 0)) / (2 * jnp.pi), "k-", lw=1.5, alpha=0.6, label="Semicircle law")
ax_z.set_xlim(-zoom, zoom)
ax_z.set_xlabel("Eigenvalue")
ax_z.set_ylabel("Density")
ax_z.set_title(rf"Zoomed spectrum ($n={n}$, $\varepsilon={eps}$, strength$={strength}$)")
ax_z.legend(fontsize=8)
plt.tight_layout()
plt.savefig("../report/figures/02_eigenvalue_spectra_zoom.pdf", bbox_inches="tight")
plt.show()

print("Done.")